## Fix Chinese Pinyin Surname Parse

The existing name parser (`CreateAuthorNames`) has CJK order detection for Han/Hangul/Kana script and for Korean-romanized-with-hyphen names, but **no order detection for Chinese pinyin 2-token Latin names**. Names like `Zhang Qi`, `Zhou Yong`, `Tang Ye` currently parse as `(first=Zhang, last=Qi)` — treating the surname as a first name.

Impact from Phase 3 production analysis (`oxjobs/working/split-authors-name-parser/eval/phase3/`):
- 31.4% of Phase 3 swap-rule candidates are profiles whose `full_name` is mis-parsed this way.
- 1.39M rows in `author_names` match the conservative fix criteria.
- Near-zero FP rate on the sample (20/20 correctly Chinese).

**This notebook fixes existing `author_names.parsed_name` records.** It does NOT modify `CreateAuthorNames.ipynb` — that change needs to land separately so new names parse correctly going forward (see `oxjobs/.../parser_fix/parser_patch.md`).

**Scope of fix:**
- 2-token Latin-only raw names, no comma
- `tokens[0]` is in the 48-item unambiguous Chinese pinyin surname list
- `tokens[1]` is NOT in the same list (skip `Li Yang`/`Yang Li` genuine ambiguity)
- `parsed_name.first` and `parsed_name.last` currently match `(t0, t1)` (skip already-correct rows)

**Action:** swap `parsed_name.first` ↔ `parsed_name.last` for matched rows.

**Downstream:** After running, `authors_for_matching` needs rebuild so block_keys shift. Then the MatchAuthors pipeline will naturally re-group works on the next run.

### Cell 1: Build candidates table

Identifies all rows in `author_names` whose parse should be flipped. Materialized as a table so the fix, audit log, and verification all reference the same candidate set.

In [ ]:
CREATE OR REPLACE TABLE openalex.authors.pinyin_surname_swap_candidates AS
SELECT
  an.raw_author_name,
  an.parsed_name,
  LOWER(SPLIT(an.raw_author_name, ' ')[0]) AS t0,
  LOWER(SPLIT(an.raw_author_name, ' ')[1]) AS t1,
  -- The desired post-fix values (for audit log + MERGE)
  an.parsed_name.last  AS new_first,
  an.parsed_name.first AS new_last
FROM openalex.authors.author_names an
WHERE
  -- 2-token Latin-only, no comma
  an.raw_author_name RLIKE '^[A-Za-z][A-Za-z-]* [A-Za-z][A-Za-z-]*$'
  AND an.raw_author_name NOT LIKE '%,%'
  -- t0 in unambiguous Chinese pinyin surname list
  AND LOWER(SPLIT(an.raw_author_name, ' ')[0]) IN (
    'wang','li','zhang','liu','chen','yang','huang','zhao','zhou',
    'xu','sun','zhu','hu','guo','gao','luo',
    'zheng','liang','xie','song','tang','feng','deng','cao','peng',
    'zeng','xiao','tian','pan','yuan','jiang','dong','jia','ding',
    'shen','wei','meng','qian','yan','dai','du',
    'fu','zhong','kong','cui','qiu','lei','yao'
  )
  -- t1 NOT in that list (skip 'Li Yang'/'Yang Li' genuine ambiguity)
  AND LOWER(SPLIT(an.raw_author_name, ' ')[1]) NOT IN (
    'wang','li','zhang','liu','chen','yang','huang','zhao','zhou',
    'xu','sun','zhu','hu','guo','gao','luo',
    'zheng','liang','xie','song','tang','feng','deng','cao','peng',
    'zeng','xiao','tian','pan','yuan','jiang','dong','jia','ding',
    'shen','wei','meng','qian','yan','dai','du',
    'fu','zhong','kong','cui','qiu','lei','yao'
  )
  -- Current parse must be Western-order (defensive — skip already-flipped or Korean-rule-processed)
  AND LOWER(an.parsed_name.first) = LOWER(SPLIT(an.raw_author_name, ' ')[0])
  AND LOWER(an.parsed_name.last)  = LOWER(SPLIT(an.raw_author_name, ' ')[1])


### Cell 2: Validation statistics

In [ ]:
SELECT
  COUNT(*) AS total_candidates,
  COUNT(DISTINCT t0) AS distinct_surnames,
  COUNT(DISTINCT t1) AS distinct_given_names
FROM openalex.authors.pinyin_surname_swap_candidates

### Cell 3: Spot-check sample (manual review)

50 random rows. Each should show a clear Chinese name where the flip produces the correct (first, last). Look for: Hispanic initials like `MA Prieto-Palomino` (shouldn't occur — `ma` is excluded from the unambiguous list), Western given names + surname (also shouldn't occur since t0 must be a surname).

In [ ]:
SELECT raw_author_name,
  parsed_name.first AS current_first,
  parsed_name.last  AS current_last,
  new_first,
  new_last
FROM openalex.authors.pinyin_surname_swap_candidates
ORDER BY RAND()
LIMIT 50

### Cell 4: Create audit log (for rollback)

Snapshot of previous parsed_name values before the fix. Keyed on raw_author_name. Use Cell 7's template to roll back if a systematic FP class is discovered.

In [ ]:
CREATE OR REPLACE TABLE openalex.authors.pinyin_surname_swap_log AS
SELECT
  raw_author_name,
  parsed_name AS previous_parsed_name,
  parsed_name.first AS previous_first,
  parsed_name.last  AS previous_last,
  new_first,
  new_last,
  current_timestamp() AS fix_timestamp
FROM openalex.authors.pinyin_surname_swap_candidates

### Cell 5: Execute the fix

**WARNING**: This overwrites `parsed_name` on `openalex.authors.author_names` for the ~1.39M candidate rows. Verify Cells 2–3 before running.

Flips `first` ↔ `last` while preserving `title`, `middle`, `suffix`, `nickname` (all should be NULL for 2-token names, but we preserve the struct shape defensively).

In [ ]:
MERGE INTO openalex.authors.author_names AS target
USING openalex.authors.pinyin_surname_swap_candidates AS source
ON target.raw_author_name = source.raw_author_name
WHEN MATCHED THEN UPDATE SET
  target.parsed_name = named_struct(
    'title',    target.parsed_name.title,
    'first',    target.parsed_name.last,
    'middle',   target.parsed_name.middle,
    'last',     target.parsed_name.first,
    'suffix',   target.parsed_name.suffix,
    'nickname', target.parsed_name.nickname
  )

### Cell 6: Post-fix verification

Confirms `parsed_name.first` now equals `new_first` (the pre-fix `parsed_name.last`) for all logged rows. `still_old` should be 0.

In [ ]:
SELECT
  COUNT(*) AS logged_rows,
  SUM(CASE WHEN LOWER(an.parsed_name.first) = LOWER(src.new_first) THEN 1 ELSE 0 END) AS now_flipped,
  SUM(CASE WHEN LOWER(an.parsed_name.first) = LOWER(src.previous_first) THEN 1 ELSE 0 END) AS still_old
FROM openalex.authors.pinyin_surname_swap_log src
JOIN openalex.authors.author_names an ON an.raw_author_name = src.raw_author_name

### Cell 7: Downstream — surface author profiles whose block_key will shift

Authors whose `full_name` is in the fixed set need `authors_for_matching` rebuild so their block_key reflects the new (first, last). After that rebuild, MatchAuthors will naturally re-group works correctly on its next scheduled run.

In [ ]:
SELECT
  COUNT(DISTINCT a.id) AS author_profiles_affected,
  COUNT(DISTINCT a.full_name) AS distinct_full_names_fixed
FROM openalex.authors.authors a
JOIN openalex.authors.pinyin_surname_swap_log src
  ON a.full_name = src.raw_author_name

### Cell 8: Rollback template (use only if a systematic FP class is discovered)

```sql
MERGE INTO openalex.authors.author_names AS target
USING openalex.authors.pinyin_surname_swap_log AS source
ON target.raw_author_name = source.raw_author_name
WHEN MATCHED THEN UPDATE SET
  target.parsed_name = source.previous_parsed_name
```

Add a `WHERE` clause inside the USING subquery to restrict rollback to a specific class (e.g., a specific surname or raw-name pattern). Don't run as-is — it reverses all fixes.